# Knowledge Distillation

## Learning Objectives
1. Understand temperature scaling and why soft targets carry richer information than hard labels.
2. Implement the KL-divergence distillation loss from scratch in NumPy.
3. Train a teacher-student pair in PyTorch with the full KD loss (hard + soft) and OOM handling.
4. Apply intermediate-layer hint loss, self-distillation, and BERT compression patterns for production model compression.

In [ ]:
import numpy as np
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import DataLoader, TensorDataset

np.random.seed(42)
torch.manual_seed(42)
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Using device: {device}')


## Level 1: Temperature Scaling and KL Divergence (NumPy)

In [ ]:
# Knowledge distillation transfers knowledge via soft probability distributions.
# Temperature T > 1 flattens the teacher's distribution, revealing inter-class
# relationships that hard one-hot labels discard.

def softmax_temp(logits, T=1.0):
    '''Apply temperature-scaled softmax.
    Args:
        logits: raw output scores, shape (n_classes,)
        T: temperature; T>1 softens, T<1 sharpens distribution
    Returns:
        probability distribution over classes
    '''
    shifted = logits / T - np.max(logits / T)   # numerical stability
    exp_vals = np.exp(shifted)
    return exp_vals / exp_vals.sum()

def kl_divergence(p, q, eps=1e-9):
    '''KL(p || q) — how much q diverges from p.
    In distillation: p=teacher soft probs, q=student soft probs.
    '''
    return np.sum(p * np.log((p + eps) / (q + eps)))

# Simulate teacher logits for a 5-class problem
teacher_logits = np.array([3.2, 1.0, 0.5, -0.8, -1.5])

print('Effect of temperature on teacher soft targets:')
for T in [0.5, 1.0, 2.0, 5.0, 10.0]:
    probs = softmax_temp(teacher_logits, T)
    entropy = -np.sum(probs * np.log(probs + 1e-9))
    print(f'  T={T:4.1f}  probs={np.round(probs, 3)}  entropy={entropy:.3f}')

# KL loss between teacher and student (both at temperature T)
student_logits = np.array([2.9, 1.2, 0.3, -0.5, -1.8])
T = 4.0
p_teacher = softmax_temp(teacher_logits, T)
p_student = softmax_temp(student_logits, T)
kl_loss = kl_divergence(p_teacher, p_student) * (T ** 2)  # T^2 rescales gradient
print(f'\nKL distillation loss at T={T}: {kl_loss:.6f}')
print('T^2 rescaling keeps gradient magnitude consistent across temperatures.')


## Level 2: Teacher-Student Training in PyTorch (KD Loss + OOM Handling)

In [ ]:
# Full knowledge distillation training loop.
# Total loss = alpha * CE(student, hard_labels) + (1-alpha) * T^2 * KL(teacher, student)
# Teacher is frozen; only student updates.

torch.manual_seed(0)
N_CLASSES = 10

# Synthetic dataset: 1000 samples, 32 features
X_data = torch.randn(1000, 32, device=device)
y_data = torch.randint(0, N_CLASSES, (1000,), device=device)
train_ds = TensorDataset(X_data[:800], y_data[:800])
val_ds   = TensorDataset(X_data[800:], y_data[800:])
train_loader = DataLoader(train_ds, batch_size=64, shuffle=True)
val_loader   = DataLoader(val_ds,   batch_size=128)


class Teacher(nn.Module):
    '''Large teacher: 3 hidden layers.'''
    def __init__(self):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(32, 256), nn.ReLU(),
            nn.Linear(256, 128), nn.ReLU(),
            nn.Linear(128, N_CLASSES)
        )
    def forward(self, x): return self.net(x)


class Student(nn.Module):
    '''Compact student: 1 hidden layer (~4x fewer params).'''
    def __init__(self):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(32, 64), nn.ReLU(),
            nn.Linear(64, N_CLASSES)
        )
    def forward(self, x): return self.net(x)


# Pre-train teacher for 30 epochs
teacher = Teacher().to(device)
opt_t = torch.optim.Adam(teacher.parameters(), lr=1e-3)
for _ in range(30):
    for xb, yb in train_loader:
        opt_t.zero_grad()
        F.cross_entropy(teacher(xb), yb).backward()
        opt_t.step()
teacher.eval()
for p in teacher.parameters(): p.requires_grad_(False)  # freeze teacher

# Distillation training
student = Student().to(device)
opt_s   = torch.optim.Adam(student.parameters(), lr=1e-3)
ALPHA   = 0.3   # weight for hard-label CE loss
T_DIST  = 4.0   # distillation temperature
history = []

for epoch in range(40):
    student.train()
    for xb, yb in train_loader:
        try:
            opt_s.zero_grad()
            s_logits = student(xb)
            with torch.no_grad():
                t_logits = teacher(xb)
            # Hard label loss
            hard_loss = F.cross_entropy(s_logits, yb)
            # Soft label KL loss (T^2 rescales gradient magnitude)
            soft_loss = F.kl_div(
                F.log_softmax(s_logits / T_DIST, dim=-1),
                F.softmax(t_logits / T_DIST, dim=-1),
                reduction='batchmean'
            ) * (T_DIST ** 2)
            loss = ALPHA * hard_loss + (1 - ALPHA) * soft_loss
            loss.backward()
            opt_s.step()
        except RuntimeError as exc:
            if 'out of memory' in str(exc).lower():
                torch.cuda.empty_cache()
                print('OOM — reduce batch_size')
            else:
                raise
    student.eval()
    with torch.no_grad():
        correct = sum(
            (student(xb).argmax(1) == yb).sum().item()
            for xb, yb in val_loader
        )
    history.append(correct / len(val_ds))

print(f'Student (distilled) val accuracy: {history[-1]:.4f}')
t_params = sum(p.numel() for p in teacher.parameters())
s_params = sum(p.numel() for p in student.parameters())
print(f'Teacher params: {t_params:,}   Student params: {s_params:,}  '
      f'Compression ratio: {t_params/s_params:.1f}x')


## Real-World Example 1: Intermediate Layer Hint Loss

In [ ]:
# FitNets-style hint loss: student's intermediate feature map should
# match a linear projection of the teacher's intermediate features.
# Adds supervision signal deeper in the network beyond just the final logits.

torch.manual_seed(1)


class TeacherHint(nn.Module):
    '''Teacher with accessible intermediate layer.'''
    def __init__(self):
        super().__init__()
        self.layer1 = nn.Sequential(nn.Linear(32, 256), nn.ReLU())
        self.layer2 = nn.Sequential(nn.Linear(256, 128), nn.ReLU())
        self.head   = nn.Linear(128, N_CLASSES)
    def forward(self, x):
        h1 = self.layer1(x)
        h2 = self.layer2(h1)
        return self.head(h2), h2   # (logits, hint_features)


class StudentHint(nn.Module):
    '''Student with guided layer + projection to match teacher hint dim.'''
    def __init__(self):
        super().__init__()
        self.layer1  = nn.Sequential(nn.Linear(32, 64), nn.ReLU())
        self.project = nn.Linear(64, 128)   # maps to teacher hint dim
        self.head    = nn.Linear(64, N_CLASSES)
    def forward(self, x):
        h = self.layer1(x)
        return self.head(h), self.project(h)


t_hint = TeacherHint().to(device)
s_hint = StudentHint().to(device)

# Pre-train teacher briefly
opt_th = torch.optim.Adam(t_hint.parameters(), lr=1e-3)
for _ in range(20):
    for xb, yb in train_loader:
        opt_th.zero_grad()
        F.cross_entropy(t_hint(xb)[0], yb).backward()
        opt_th.step()
t_hint.eval()
for p in t_hint.parameters(): p.requires_grad_(False)

opt_sh = torch.optim.Adam(s_hint.parameters(), lr=1e-3)
LAMBDA_HINT = 0.5   # hint loss weight

for epoch in range(30):
    s_hint.train()
    for xb, yb in train_loader:
        opt_sh.zero_grad()
        s_logits, s_feat = s_hint(xb)
        with torch.no_grad():
            t_logits, t_feat = t_hint(xb)
        ce_loss   = F.cross_entropy(s_logits, yb)
        hint_loss = F.mse_loss(s_feat, t_feat)    # match intermediate features
        total     = ce_loss + LAMBDA_HINT * hint_loss
        total.backward(); opt_sh.step()

s_hint.eval()
with torch.no_grad():
    correct = sum(
        (s_hint(xb)[0].argmax(1) == yb).sum().item()
        for xb, yb in val_loader
    )
print(f'Student (hint loss) val accuracy: {correct / len(val_ds):.4f}')
print(f'LAMBDA_HINT={LAMBDA_HINT}: hint loss guides intermediate representation.')


## Real-World Example 2: Self-Distillation (Born-Again Networks)

In [ ]:
# Self-distillation: distill a model into a model of the SAME architecture.
# Born-Again Networks show that even identical-capacity students outperform
# the original because the soft targets act as regularisation.

torch.manual_seed(2)


def train_baseline(loader, n_epochs=35):
    '''Train a fresh Student with hard labels only.'''
    m   = Student().to(device)
    opt = torch.optim.Adam(m.parameters(), lr=1e-3)
    for _ in range(n_epochs):
        m.train()
        for xb, yb in loader:
            opt.zero_grad()
            F.cross_entropy(m(xb), yb).backward()
            opt.step()
    return m


def self_distil(teacher_model, loader, T=3.0, alpha=0.4, n_epochs=35):
    '''Distil teacher into same-architecture student.'''
    teacher_model.eval()
    for p in teacher_model.parameters(): p.requires_grad_(False)
    m   = Student().to(device)
    opt = torch.optim.Adam(m.parameters(), lr=1e-3)
    for _ in range(n_epochs):
        m.train()
        for xb, yb in loader:
            opt.zero_grad()
            s_log = m(xb)
            with torch.no_grad():
                t_log = teacher_model(xb)
            hard = F.cross_entropy(s_log, yb)
            soft = F.kl_div(
                F.log_softmax(s_log / T, dim=-1),
                F.softmax(t_log / T, dim=-1),
                reduction='batchmean'
            ) * T ** 2
            (alpha * hard + (1 - alpha) * soft).backward()
            opt.step()
    return m


gen0 = train_baseline(train_loader)
gen1 = self_distil(gen0, train_loader)
gen2 = self_distil(gen1, train_loader)


def val_acc(m):
    m.eval()
    with torch.no_grad():
        c = sum((m(xb).argmax(1) == yb).sum().item() for xb, yb in val_loader)
    return c / len(val_ds)


print(f'Generation 0 (baseline)   val accuracy: {val_acc(gen0):.4f}')
print(f'Generation 1 (self-distil) val accuracy: {val_acc(gen1):.4f}')
print(f'Generation 2 (self-distil) val accuracy: {val_acc(gen2):.4f}')
print('Each generation typically improves — soft targets regularise better than hard labels.')


## Real-World Example 3: BERT Compression via Distillation

In [ ]:
# DistilBERT-style compression: student is a 6-layer BERT trained to mimic
# a 12-layer BERT teacher using hidden state alignment + CE loss.
# Here we simulate the pattern with small transformers to run without GPU.

try:
    from transformers import (
        AutoTokenizer, AutoModelForSequenceClassification,
        DistilBertForSequenceClassification, DistilBertConfig
    )

    print('Demonstrating BERT distillation pattern (simulated — no data download):')
    print()
    print('Step 1: Load teacher (bert-base-uncased, 12 layers, 110M params)')
    print('  teacher = AutoModelForSequenceClassification.from_pretrained(')
    print('      "bert-base-uncased", num_labels=2)')
    print()
    print('Step 2: Define student (DistilBERT, 6 layers, 66M params)')
    print('  config  = DistilBertConfig(n_layers=6, n_heads=12)')
    print('  student = DistilBertForSequenceClassification(config)')
    print()
    print('Step 3: Distillation loss = CE(student, labels)')
    print('                           + alpha * KL(teacher_logits, student_logits)')
    print('                           + beta  * cosine_loss(teacher_hidden, student_hidden)')
    print()
    print('Step 4: Training loop (identical to RW2 distil() but with HuggingFace trainer):')
    print('  training_args = TrainingArguments(output_dir="./distilbert", fp16=True, ...)')
    print('  trainer = Trainer(model=student, teacher_model=teacher, ...)')
    print('  trainer.train()')
    print()
    print('Result: DistilBERT retains ~97% of BERT performance at 60% the size.')
    print('Compression ratio: 110M -> 66M params (1.67x smaller, 60% faster inference).')

except ImportError:
    pass

# Standalone simulation with tiny transformers
torch.manual_seed(3)


class TinyTransformer(nn.Module):
    '''Minimal transformer encoder for simulation.'''
    def __init__(self, d_model=64, n_heads=4, n_layers=2, n_classes=5):
        super().__init__()
        self.embed  = nn.Linear(32, d_model)
        encoder_layer = nn.TransformerEncoderLayer(
            d_model=d_model, nhead=n_heads,
            dim_feedforward=d_model*4, batch_first=True
        )
        self.encoder = nn.TransformerEncoder(encoder_layer, num_layers=n_layers)
        self.head    = nn.Linear(d_model, n_classes)
    def forward(self, x):
        x = self.embed(x.unsqueeze(1))   # (B, 1, d_model)
        return self.head(self.encoder(x).squeeze(1))


N_CLASSES_BERT = 5
X_bert = torch.randn(600, 32, device=device)
y_bert = torch.randint(0, N_CLASSES_BERT, (600,), device=device)
bert_ds   = TensorDataset(X_bert[:480], y_bert[:480])
bert_val  = TensorDataset(X_bert[480:], y_bert[480:])
bert_trl  = DataLoader(bert_ds,  batch_size=32, shuffle=True)
bert_vll  = DataLoader(bert_val, batch_size=64)

big_teacher = TinyTransformer(d_model=128, n_heads=4, n_layers=4,
                              n_classes=N_CLASSES_BERT).to(device)
small_student = TinyTransformer(d_model=64, n_heads=4, n_layers=2,
                               n_classes=N_CLASSES_BERT).to(device)

# Quick teacher pre-training
opt_bt = torch.optim.Adam(big_teacher.parameters(), lr=5e-4)
for _ in range(20):
    for xb, yb in bert_trl:
        opt_bt.zero_grad()
        F.cross_entropy(big_teacher(xb), yb).backward()
        opt_bt.step()
big_teacher.eval()
for p in big_teacher.parameters(): p.requires_grad_(False)

# Distil big_teacher -> small_student
opt_bs = torch.optim.Adam(small_student.parameters(), lr=5e-4)
T_BERT = 3.0
for _ in range(20):
    small_student.train()
    for xb, yb in bert_trl:
        opt_bs.zero_grad()
        sl = small_student(xb)
        with torch.no_grad(): tl = big_teacher(xb)
        loss = (0.3 * F.cross_entropy(sl, yb)
                + 0.7 * F.kl_div(F.log_softmax(sl/T_BERT, -1),
                                 F.softmax(tl/T_BERT, -1),
                                 reduction='batchmean') * T_BERT**2)
        loss.backward(); opt_bs.step()

small_student.eval()
with torch.no_grad():
    correct = sum((small_student(xb).argmax(1)==yb).sum().item()
                  for xb, yb in bert_vll)
t_p = sum(p.numel() for p in big_teacher.parameters())
s_p = sum(p.numel() for p in small_student.parameters())
print(f'Teacher params: {t_p:,}   Student params: {s_p:,}  '
      f'Compression: {t_p/s_p:.1f}x')
print(f'Distilled student val accuracy: {correct/len(bert_val):.4f}')
